# Large-scale OR: RAG-only

Two-stage GPT-4.1 workflow without worked cases. NRM uses the original planned CSVQA tool through a ReAct agent; RA, TP, AP, and FLP use their original FAISS CSVQA path. The Others route retains its original schema-preview, abstract-plan, and code-generation flow with the worked-case prompts removed.


In [ ]:
from __future__ import annotations

import csv, hashlib, json, math, os, re, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path(os.environ.get('LEAN_LLM_OPT_ROOT', Path.cwd())).resolve()
BENCHMARK = ROOT / 'Test_Dataset/Large-scale-or/Large-scale-or-101.csv'
MODEL = 'gpt-4.1-2025-04-14'
if not BENCHMARK.is_file():
    raise FileNotFoundError('Run from the repository root or set LEAN_LLM_OPT_ROOT')

CLASS_RULES = '''Return exactly one label: NRM, RA, TP, FLP, AP, Mixture, or Others.
NRM allocates finite inventory or capacity to demand for revenue.
RA allocates resources among activities in a primarily single-stage model.
TP is source-to-destination shipment with supply, demand, and route costs.
FLP links facility-opening decisions to assignment or shipment.
AP is one-to-one matching or assignment.
Mixture has materially coupled modeling mechanisms or multiple stages.
Others is a coherent optimization problem outside those classes.
Use the current question's objective, variables, domains, and constraints.
Output the label token alone.'''

OUTPUT_RULES = '''Return exactly two sections:
### Mathematical Model
Specify sets, parameters, variables and their domains, objective, and all constraints.
Use an LP or MILP as required by the current question.
### Python Code
Provide exactly one fenced Python code block. It must be complete gurobipy
code, create a model named m or model, call optimize(), and leave that solved
model available at top level. Read only the current case CSV paths listed in
this request. Derive identifiers, dimensions, and coefficients from these files.
The code must solve from the repository root; do not invent file paths.'''

def resolve_csv_paths(address):
    out=[]
    for raw in str(address).splitlines():
        raw=raw.strip()
        if not raw:continue
        path=Path(raw)
        path=(path if path.is_absolute() else ROOT/path).resolve()
        try:path.relative_to(ROOT)
        except ValueError as exc:raise ValueError(f'Path leaves repository: {raw}') from exc
        if path.suffix.lower()!='.csv' or not path.is_file():
            raise FileNotFoundError(path)
        out.append(path)
    if not out:raise ValueError('No CSV files listed for this case')
    return out

def parse_objective(raw):
    text=str(raw).strip().replace(',','').replace('$','')
    m=re.search(r'Optimal\s+objective\s*[:=]?\s*([-+\d.eE]+)',text,re.I)
    value=float(m.group(1) if m else text)
    if not math.isfinite(value):raise ValueError(raw)
    return value

def gold_class(raw):
    text=str(raw).strip().upper()
    if text=='UFLP':return 'FLP'
    if text.startswith('OTHERS'):return 'Others'
    if text=='MIXTURE':return 'Mixture'
    if text in {'NRM','RA','TP','FLP','AP'}:return text
    raise ValueError(raw)

def route(label):
    if label in {'Mixture','Others'}:return 'Others'
    if label not in {'NRM','RA','TP','FLP','AP'}:raise ValueError(f'Bad class: {label!r}')
    return label

def cases():
    with BENCHMARK.open(encoding='utf-8-sig',newline='') as f:
        rows=list(csv.DictReader(f))
    if len(rows)!=101:raise ValueError(f'Expected 101 cases, found {len(rows)}')
    result=[]
    for i,row in enumerate(rows):
        result.append({'row':i,'query':row['Query'],'type':row['Problem Type'],
                       'files':resolve_csv_paths(row['Dataset_address']),
                       'gold':parse_objective(row['Label-objective'])})
    return result

def rel(path):return str(path.relative_to(ROOT))

def check_files():
    data=cases()
    counts={n:sum(len(c['files'])==n for c in data) for n in (1,2,3)}
    print('Rows:',len(data),'Files per row:',counts)
    return data

def output_text(response):
    if getattr(response,'status',None)=='incomplete':
        raise RuntimeError(f'Incomplete model response: {response.incomplete_details}')
    text=str(getattr(response,'output_text','') or '').strip()
    if not text:raise RuntimeError('Empty model response')
    return text

def usage(response):
    u=getattr(response,'usage',None)
    return {'input':int(getattr(u,'input_tokens',0) or 0),
            'output':int(getattr(u,'output_tokens',0) or 0)}

def plain_call(client,system,user):
    response=client.responses.create(model=MODEL,temperature=0,max_output_tokens=32768,
        input=[{'role':'system','content':system},{'role':'user','content':user}])
    return output_text(response),[usage(response)]

def extract_code(answer):
    if '### Mathematical Model' not in answer or '### Python Code' not in answer:
        raise ValueError('Model/code sections missing')
    blocks=re.findall(r'```(?:python|py)\s*\n(.*?)\n```',answer,re.I|re.S)
    if len(blocks)!=1:raise ValueError(f'Expected one Python block, found {len(blocks)}')
    return blocks[0].strip()

SOLVER_FOOTER = '''
import gurobipy as _ab_gp
_ab_model=globals().get('m',globals().get('model'))
if not isinstance(_ab_model,_ab_gp.Model):
    raise RuntimeError('Code did not leave a Gurobi model in m or model')
if _ab_model.Status!=_ab_gp.GRB.OPTIMAL:
    raise RuntimeError(f'Gurobi status: {_ab_model.Status}')
print('ABLATION_OBJECTIVE='+repr(float(_ab_model.ObjVal)))
'''

def solve_code(source,timeout=300):
    run=subprocess.run([sys.executable,'-c',source+'\n'+SOLVER_FOOTER],
        cwd=ROOT,text=True,capture_output=True,timeout=timeout,check=False)
    transcript=(run.stdout or '')+'\n'+(run.stderr or '')
    if run.returncode:raise RuntimeError(transcript[-3000:])
    match=re.search(r'ABLATION_OBJECTIVE=([-+\d.eE]+)',transcript)
    if not match:raise RuntimeError('Objective missing: '+transcript[-3000:])
    return float(match.group(1)),transcript[-3000:]

def save_scores(folder,selected):
    latest={}
    for line in (folder/'records.jsonl').read_text(encoding='utf-8').splitlines():
        if line.strip():
            record=json.loads(line);latest[record['row']]=record
    rows=[latest[i] for i in sorted(latest)]
    fields=['run_id','method','dataset','instance_id','gold_type','pred_type',
            'correct_optimal','gold_objective','predicted_objective','status',
            'dataset_address','input_tokens','output_tokens','tool_calls','wall_s']
    with (folder/'scored.csv').open('w',encoding='utf-8',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=fields);writer.writeheader()
        for r in rows:
            writer.writerow({'run_id':folder.name,'method':METHOD,
                'dataset':'Large-Scale-OR','instance_id':r['row'],
                'gold_type':r['gold_type'],'pred_type':r.get('pred_type'),
                'correct_optimal':int(bool(r.get('correct_optimal'))),
                'gold_objective':r['gold_objective'],
                'predicted_objective':r.get('predicted_objective'),
                'status':r['status'],'dataset_address':'\n'.join(r['files']),
                'input_tokens':(r.get('usage') or {}).get('input',0),
                'output_tokens':(r.get('usage') or {}).get('output',0),
                'tool_calls':r.get('tool_calls',0),'wall_s':r['wall_s']})
    summary={'run_id':folder.name,'method':METHOD,'n':len(rows),
        'complete':set(latest)==set(selected),
        'correct_optimal':sum(bool(r.get('correct_optimal')) for r in rows),
        'classification_correct':sum(bool(r.get('classification_correct')) for r in rows),
        'failed':sum(r['status']!='ok' for r in rows),
        'input_tokens':sum((r.get('usage') or {}).get('input',0) for r in rows),
        'output_tokens':sum((r.get('usage') or {}).get('output',0) for r in rows)}
    (folder/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')


In [ ]:
# The full workflow's CSVQA implementation is embedded below.
from typing import Any

PROJECT_ROOT = ROOT
MODEL_SNAPSHOT = MODEL
EMBEDDING_MODEL = 'text-embedding-ada-002'
CSVQA_MODE_BY_ROUTE = {
    'NRM': 'planned', 'RA': 'legacy', 'TP': 'legacy',
    'AP': 'legacy', 'FLP': 'legacy', 'Others': 'legacy',
}

def require_rag_dependencies():
    import pandas as pd
    from langchain_classic.agents import AgentType, Tool, initialize_agent
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain
    from langchain_community.vectorstores import FAISS
    from langchain_core.documents import Document
    from langchain_core.messages import HumanMessage
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
    globals().update(locals())

def make_llm():
    return ChatOpenAI(model_name=MODEL_SNAPSHOT,
        openai_api_key=os.environ['OPENAI_API_KEY'],
        temperature=0.0, top_p=1, n=1, max_retries=0, timeout=180)

def make_embeddings():
    return OpenAIEmbeddings(model=EMBEDDING_MODEL,
        openai_api_key=os.environ['OPENAI_API_KEY'], max_retries=0)

FILTER_OPERATORS = {
    "exact", "prefix", "contains", "in", "not_in", "eq", "ne",
    "gt", "ge", "lt", "le", "between", "is_null", "not_null",
}

FILTER_DTYPES = {"string", "number", "date"}

CSVQA_LEGACY_SYSTEM_PROMPT = (
    "Retrieve every CSV row needed by the query in source order. Preserve exact identifiers and "
    "numeric values. If the query names a subset, retrieve that complete subset; otherwise retrieve "
    "all data. Use only the supplied context: {context}"
)

CSVQA_LEGACY_TOOL_DESCRIPTION = (
    "Retrieve complete source-ordered CSV evidence needed to formulate the optimization problem."
)

CSVQA_PLANNED_PROMPTS = {'NRM': 'Select every source row and column needed to define the revenue-management objective, decision entities, demand limits, inventory or capacity limits, and query-specific bounds. Infer exact columns from the query and schema. Filter only an explicit subset.'}

CSVQA_TOOL_DESCRIPTIONS = {'NRM': 'Return the canonical CSV rows, columns, filters, and relationships needed by the query.'}

def normalize_route(value: Any) -> str:
    text = str(value or "").strip().upper()
    aliases = {
        "NETWORK REVENUE MANAGEMENT": "NRM", "NETWORK REVENUE MANAGEMENT PROBLEM": "NRM",
        "RESOURCE ALLOCATION": "RA", "RESOURCE ALLOCATION PROBLEM": "RA",
        "TRANSPORTATION": "TP", "TRANSPORTATION PROBLEM": "TP",
        "ASSIGNMENT": "AP", "ASSIGNMENT PROBLEM": "AP",
        "FACILITY LOCATION": "FLP", "FACILITY LOCATION PROBLEM": "FLP",
    }
    text = aliases.get(text, text)
    if text in {"MIXTURE", "OTHER", "OTHERS"} or text.startswith("OTHERS"):
        return "Others"
    if text == "UFLP":
        return "FLP"
    if text in {"NRM", "RA", "TP", "AP", "FLP"}:
        return text
    raise ValueError(f"Unsupported execution route: {value!r}")

def read_csv_compat(path, **kwargs):
    """Read a project-relative or absolute CSV with the requested pandas options."""
    path = Path(path).expanduser()
    path = path if path.is_absolute() else PROJECT_ROOT / path
    last_error = None
    for encoding in ("utf-8-sig", "utf-8", "gbk", "latin-1"):
        try:
            return pd.read_csv(path, encoding=encoding, **kwargs)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise UnicodeError(f"Unable to decode CSV: {path}") from last_error

def _read_csv(path):
    """Read values as text and remove only empty structure or a clear export index."""
    frame = read_csv_compat(path, dtype=str, keep_default_na=False)
    frame.columns = [str(column).lstrip("\ufeff") for column in frame.columns]
    if frame.empty:
        return frame
    nonempty = frame.apply(lambda column: column.astype(str).str.strip().ne(""))
    frame = frame.loc[nonempty.any(axis=1), nonempty.any(axis=0)].copy()
    for column in list(frame.columns):
        values = frame[column].astype(str).tolist()
        sequential = values in (
            [str(i) for i in range(len(values))],
            [str(i) for i in range(1, len(values) + 1)],
        )
        if sequential and re.fullmatch(r"(?:unnamed(?::\s*\d+)?|index)", column.strip(), re.I):
            frame = frame.drop(columns=column)
    return frame

def _load_tables(dataset_address):
    """Resolve one-or-more newline-separated CSV paths in source order."""
    tables = []
    for raw in str(dataset_address or "").splitlines():
        raw = raw.strip()
        if not raw:
            continue
        path = Path(raw).expanduser()
        path = path if path.is_absolute() else (PROJECT_ROOT / path).resolve()
        if not path.is_file():
            raise FileNotFoundError(f"CSVQA cannot find dataset file: {path}")
        tables.append({"file_index": len(tables), "path": path, "frame": _read_csv(path)})
    if not tables:
        raise ValueError("CSVQA requires at least one CSV path")
    return tables

def _build_profile(tables, query):
    """Expose schema, compact samples, and evidence for query-named terms."""
    quote_pattern = r'"([^\"]+)"|\'([^\']+)\'|“([^”]+)”|‘([^’]+)’'
    terms = list(dict.fromkeys(
        next(value for value in match.groups() if value is not None).strip()
        for match in re.finditer(quote_pattern, str(query or ""))
    ))
    key = lambda value: "".join(ch for ch in str(value or "").casefold() if ch.isalnum())
    profile = []
    for table in tables:
        frame = table["frame"]
        term_matches = {}
        for term in terms:
            wanted, value_matches = term.casefold(), {}
            for column in frame.columns:
                values = frame[column].astype(str).str.strip().str.casefold()
                counts = {
                    "exact": int(values.eq(wanted).sum()),
                    "prefix": int(values.str.startswith(wanted).sum()),
                    "contains": int(values.str.contains(wanted, regex=False).sum()),
                }
                if counts["contains"]:
                    value_matches[str(column)] = counts
            term_matches[term] = {
                "column_matches": [str(column) for column in frame.columns if key(column) == key(term)],
                "value_matches": value_matches,
            }
        profile.append({
            "file_index": table["file_index"],
            "file_name": table["path"].name,
            "rows": len(frame),
            "columns": list(frame.columns),
            "sample_values": {
                str(column): frame[column].astype(str).loc[lambda values: values.str.strip().ne("")]
                .drop_duplicates().head(4).tolist()
                for column in frame.columns
            },
            "query_terms": term_matches,
        })
    return profile

def _ask_extraction_plan(llm, route, query, tables, route_prompt):
    """Ask for one declarative plan and return the parsed plan plus raw response."""
    profile = json.dumps(_build_profile(tables, query), ensure_ascii=False)
    request = f"""You are a CSV extraction planner. Return one JSON object only.
Choose tables, columns, and filters from the user query and the supplied CSV profile. Do not return
data, Python, formulas, prose, joins, aggregation, sorting, sampling, or invented fields.

Shared plan schema:
{{"route":"{route}","tables":[{{"file_index":0,"role":"descriptive non-unique role",
"columns":["exact column"] or "*","filters":{{"logic":"and|or","conditions":[
{{"column":"exact column","operator":"exact|prefix|contains|in|not_in|eq|ne|gt|ge|lt|le|between|is_null|not_null",
"dtype":"string|number|date","value":"scalar or list","evidence":"exact query span",
"format":"optional date format","inclusive":"both|left|right|neither"}}]}}}}],
"ignored_file_indices":[],"relationships":[{{"type":"matrix",
"matrix_table_id":"file_2_view_0","row_id_column":"exact column",
"row_axis":{{"table_id":"file_0_view_0","id_column":"exact column"}},
"column_axis":{{"table_id":"file_1_view_0","id_column":"exact column"}}}}]}}

Python assigns each logical view a table_id of file_<file_index>_view_<zero-based occurrence for that
file>. A physical file may appear in multiple table specifications and roles may repeat. Every input
file must appear in tables or ignored_file_indices. Ignore a file only when the query does not need it.
Every explicit subset must have a filter and every filter must quote supporting query evidence.
Declare a matrix relationship only when the profile exposes its two axes unambiguously. When one label
column contains both decision entities and scalar parameter rows, retain every row that supplies a
query-required coefficient, price, cost, capacity, or bound; use separate logical views when helpful.

Route guidance:
{route_prompt}

User query:
{query}

CSV profile:
{profile}
"""
    text = str(llm.invoke([HumanMessage(content=request)]).content or "").strip()
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.I) if text.startswith("```") else text
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start < 0 or end < start:
        error = ValueError(f"CSVQA planner did not return a JSON object: {text[:200]}")
        error.planner_output = text
        raise error
    try:
        plan = json.loads(cleaned[start:end + 1])
    except ValueError as exc:
        error = ValueError(f"CSVQA planner returned invalid JSON: {exc}")
        error.planner_output = text
        raise error from exc
    if not isinstance(plan, dict):
        raise ValueError("CSVQA extraction plan must be a JSON object")
    return plan, text

def _apply_condition(series, condition):
    """Apply one allowlisted string, number, date, range, set, or null predicate."""
    operator, dtype = condition["operator"], condition.get("dtype", "string")
    raw, missing = series.astype(str).str.strip(), series.astype(str).str.strip().eq("")
    if operator in {"is_null", "not_null"}:
        return missing if operator == "is_null" else ~missing
    if dtype == "number":
        values, convert = pd.to_numeric(raw, errors="coerce"), float
    elif dtype == "date":
        date_format = condition.get("format") or None
        values = pd.to_datetime(raw, format=date_format, errors="coerce")
        convert = lambda value: pd.to_datetime(value, format=date_format)
    else:
        values = raw.str.casefold()
        convert = lambda value: str(value).strip().casefold()
    if dtype != "string" and values.loc[~missing].isna().any():
        raise ValueError(f"Cannot reliably parse {series.name!r} as {dtype}")
    value = condition.get("value")
    wanted = [convert(item) for item in value] if isinstance(value, list) else convert(value)
    if operator in {"in", "not_in", "between"} and not isinstance(wanted, list):
        raise ValueError(f"Operator {operator!r} requires a list value")
    if operator == "between" and len(wanted) != 2:
        raise ValueError("Operator 'between' requires exactly two bounds")
    operations = {
        "exact": lambda: values.eq(wanted), "eq": lambda: values.eq(wanted),
        "ne": lambda: values.ne(wanted) & ~missing,
        "prefix": lambda: values.str.startswith(wanted, na=False),
        "contains": lambda: values.str.contains(wanted, regex=False, na=False),
        "in": lambda: values.isin(wanted), "not_in": lambda: ~values.isin(wanted) & ~missing,
        "gt": lambda: values.gt(wanted), "ge": lambda: values.ge(wanted),
        "lt": lambda: values.lt(wanted), "le": lambda: values.le(wanted),
        "between": lambda: values.between(
            wanted[0], wanted[1], inclusive=condition.get("inclusive", "both")
        ),
    }
    if operator not in operations:
        raise ValueError(f"Unsupported non-null filter operator: {operator}")
    return operations[operator]()

def _execute_plan(plan, tables, query, route):
    """Validate and execute a query-driven plan; validate only declared relationships."""
    key = lambda value: "".join(ch for ch in str(value or "").casefold() if ch.isalnum())
    route = normalize_route(route)
    if normalize_route(plan.get("route")) != route:
        raise ValueError(f"CSVQA plan route does not match {route}")
    specs = plan.get("tables")
    if not isinstance(specs, list) or not specs:
        raise ValueError("CSVQA plan requires a non-empty tables list")
    ignored = plan.get("ignored_file_indices") or []
    if not isinstance(ignored, list) or any(isinstance(i, bool) or not isinstance(i, int) for i in ignored):
        raise ValueError("ignored_file_indices must be a list of integers")
    by_index = {table["file_index"]: table for table in tables}
    used = {spec.get("file_index") for spec in specs}
    if used & set(ignored) or (used | set(ignored)) != set(by_index):
        raise ValueError("Every input file must be used or explicitly ignored exactly once")

    payload_tables, payload_by_id, used_conditions, view_counts = [], {}, [], {}
    for spec in specs:
        file_index, role = spec.get("file_index"), str(spec.get("role") or "data").strip()
        if isinstance(file_index, bool) or not isinstance(file_index, int) or file_index not in by_index:
            raise ValueError(f"Invalid CSV file_index: {file_index!r}")
        occurrence = view_counts.get(file_index, 0)
        view_counts[file_index] = occurrence + 1
        table_id = f"file_{file_index}_view_{occurrence}"
        table, frame = by_index[file_index], by_index[file_index]["frame"]
        requested = spec.get("columns", "*")
        columns = list(frame.columns) if requested == "*" or requested == ["*"] else requested
        if not isinstance(columns, list) or not columns or len(columns) != len(set(columns)):
            raise ValueError(f"Table {table_id!r} requires unique columns or '*'")
        unknown = [column for column in columns if column not in frame.columns]
        if unknown:
            raise ValueError(f"Unknown columns for table {table_id!r}: {unknown}")
        filters = spec.get("filters") or {}
        if not isinstance(filters, dict):
            raise ValueError(f"Filters for table {table_id!r} must be an object")
        logic, conditions = str(filters.get("logic", "and")).casefold(), filters.get("conditions") or []
        if not isinstance(conditions, list):
            raise ValueError(f"Invalid filter logic for table {table_id!r}")
        # A common schema slip puts the sole condition's operator in ``logic``.
        # It is unambiguous for one condition; multi-condition plans remain strict.
        if logic not in {"and", "or"}:
            sole_operator = str(conditions[0].get("operator", "")).casefold() if len(conditions) == 1 and isinstance(conditions[0], dict) else ""
            if logic != sole_operator:
                raise ValueError(f"Invalid filter logic for table {table_id!r}")
            logic = "and"
            filters = {**filters, "logic": logic}
        combined = None
        for condition in conditions:
            if not isinstance(condition, dict):
                raise ValueError(f"Invalid filter for table {table_id!r}: {condition!r}")
            column, operator = condition.get("column"), condition.get("operator")
            dtype, evidence = condition.get("dtype", "string"), str(condition.get("evidence") or "")
            if column not in frame.columns or operator not in FILTER_OPERATORS or dtype not in FILTER_DTYPES:
                raise ValueError(f"Invalid filter for table {table_id!r}: {condition}")
            raw_values = condition.get("value", [])
            raw_values = raw_values if isinstance(raw_values, list) else [raw_values]
            value_supported = bool(raw_values) and all(key(value) in key(query) for value in raw_values)
            if not evidence.strip() or (key(evidence) not in key(query) and not value_supported):
                raise ValueError(f"Filter lacks query evidence: {condition}")
            mask = _apply_condition(frame[column], condition)
            combined = mask if combined is None else (combined & mask if logic == "and" else combined | mask)
            used_conditions.append(condition)
        selected = frame if combined is None else frame.loc[combined]
        if selected.empty:
            raise ValueError(f"CSVQA plan returned zero rows for table {table_id!r}")
        records = [
            {"source_row": int(index), "values": {column: str(row[column]) for column in columns}}
            for index, row in selected.iterrows()
        ]
        payload = {
            "table_id": table_id, "file_index": file_index, "file_name": table["path"].name,
            "role": role or "data", "columns": columns, "original_rows": len(frame),
            "returned_rows": len(records), "filters": filters, "records": records,
        }
        payload_tables.append(payload)
        payload_by_id[table_id] = payload

    profile = _build_profile(tables, query)
    required_values = {
        term for file_profile in profile for term, matches in file_profile["query_terms"].items()
        if matches["value_matches"] and not matches["column_matches"]
    }
    condition_text = json.dumps(used_conditions, ensure_ascii=False)
    missing_filters = sorted(term for term in required_values if key(term) not in key(condition_text))
    if missing_filters:
        raise ValueError(f"Plan omitted explicit query subset filters: {missing_filters}")
    required_columns = {
        column for file_profile in profile for matches in file_profile["query_terms"].values()
        for column in matches["column_matches"]
    }
    selected_columns = {column for table in payload_tables for column in table["columns"]}
    missing_columns = sorted(required_columns - selected_columns)
    if missing_columns:
        raise ValueError(f"Plan omitted columns named by the query: {missing_columns}")

    relationships = plan.get("relationships") or []
    if not isinstance(relationships, list):
        raise ValueError("relationships must be a list")
    validated_relationships, matrix_checks = [], []
    for relationship in relationships:
        if not isinstance(relationship, dict) or relationship.get("type") != "matrix":
            raise ValueError(f"Unsupported relationship: {relationship!r}")
        matrix_id = relationship.get("matrix_table_id")
        row_axis_spec, column_axis_spec = relationship.get("row_axis") or {}, relationship.get("column_axis") or {}
        row_axis_id, column_axis_id = row_axis_spec.get("table_id"), column_axis_spec.get("table_id")
        if any(table_id not in payload_by_id for table_id in (matrix_id, row_axis_id, column_axis_id)):
            raise ValueError(f"Relationship references an unknown table_id: {relationship}")
        matrix_table, row_axis, column_axis = (
            payload_by_id[matrix_id], payload_by_id[row_axis_id], payload_by_id[column_axis_id]
        )
        row_id_column = relationship.get("row_id_column")
        row_axis_column, column_axis_column = row_axis_spec.get("id_column"), column_axis_spec.get("id_column")
        for table, column in (
            (matrix_table, row_id_column), (row_axis, row_axis_column), (column_axis, column_axis_column)
        ):
            if column not in table["columns"]:
                raise ValueError(f"Matrix ID column {column!r} is not selected for {table['table_id']!r}")

        def axis_keys(values, axis_name):
            normalized = [key(value) for value in values]
            if any(not value for value in normalized) or len(set(normalized)) != len(normalized):
                raise ValueError(f"Matrix {axis_name} IDs must be non-empty and unique: {values}")
            return normalized

        row_ids = [record["values"][row_axis_column] for record in row_axis["records"]]
        column_ids = [record["values"][column_axis_column] for record in column_axis["records"]]
        matrix_row_ids = [record["values"][row_id_column] for record in matrix_table["records"]]
        matrix_columns = [column for column in matrix_table["columns"] if column != row_id_column]
        checks = {
            "matrix_table_id": matrix_id,
            "shape": [len(matrix_table["records"]), len(matrix_columns)],
            "expected_shape": [len(row_ids), len(column_ids)],
            "row_ids_aligned": axis_keys(matrix_row_ids, "matrix row") == axis_keys(row_ids, "row axis"),
            "column_ids_aligned": axis_keys(matrix_columns, "matrix column") == axis_keys(column_ids, "column axis"),
        }
        if checks["shape"] != checks["expected_shape"] or not checks["row_ids_aligned"] or not checks["column_ids_aligned"]:
            raise ValueError(f"Matrix validation failed: {checks}")
        validated_relationships.append(relationship)
        matrix_checks.append(checks)

    payload_tables.sort(key=lambda table: (table["file_index"], table["table_id"]))
    validation = {"status": "OK", "matrix_checks": matrix_checks}
    return {
        "route": route, "query": str(query or ""), "tables": payload_tables,
        "relationships": validated_relationships, "ignored_file_indices": sorted(ignored),
        "validation": validation,
    }

def load_csv_documents(dataset_address):
    """Build one legacy document per source row in deterministic source order."""
    documents, source_order = [], 0
    for table in _load_tables(dataset_address):
        for source_row, row in table["frame"].iterrows():
            values = {str(column): str(row[column]) for column in row.index}
            documents.append(Document(
                page_content=json.dumps({"source_row": int(source_row), "values": values}, ensure_ascii=False),
                metadata={"source": str(table["path"]), "source_order": source_order},
            ))
            source_order += 1
    return documents

def build_csvqa_components(
    dataset_address, system_prompt, tool_description, *, route, user_query="",
    result_sink=None,
):
    """Build the shared CSVQA tool in query-planned or legacy FAISS mode."""
    route = normalize_route(route)
    planned = CSVQA_MODE_BY_ROUTE[route] == "planned"
    llm = make_llm()

    state = result_sink if result_sink is not None else {}

    def rag_answer(tool_query):
        documents = load_csv_documents(dataset_address)
        embeddings = make_embeddings()
        retriever = FAISS.from_documents(documents, embeddings).as_retriever(
            search_kwargs={"k": len(documents)}
        )
        retrieved = retriever.invoke(tool_query)
        source_order = [document.metadata.get("source_order") for document in retrieved]
        if len(retrieved) != len(documents) or sorted(source_order) != list(range(len(documents))):
            raise RuntimeError(
                "Legacy CSVQA retrieval did not return every source row exactly once"
            )
        retrieved = sorted(retrieved, key=lambda doc: doc.metadata["source_order"])
        prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            (
                "human",
                "Original user query (the only authority for subsets):\n{user_query}\n\n"
                "Agent retrieval request (must not narrow beyond the original query):\n{input}",
            ),
        ])
        result = create_stuff_documents_chain(llm, prompt).invoke({
            "input": tool_query, "user_query": user_query, "context": retrieved,
        })
        if isinstance(result, dict):
            metadata = result.get("response_metadata") or result.get("metadata") or {}
            finish_reason = str(
                result.get("finish_reason") or metadata.get("finish_reason") or ""
            ).casefold()
            if finish_reason in {"length", "max_tokens", "max_output_tokens"}:
                raise RuntimeError("Legacy CSVQA model output was truncated by the service")
            answer = result.get("answer", result.get("output", ""))
        else:
            answer = result
        if not str(answer or "").strip():
            raise RuntimeError("Legacy CSVQA model returned an empty result")
        return str(answer)

    def qa_wrapper(tool_query):
        plan, planner_outputs, errors, status, fallback_reason = None, [], [], "LEGACY_FAISS", None
        attempt_count = 0
        if planned:
            tables = _load_tables(dataset_address)
            attempt_count = 1
            raw_output = ""
            try:
                plan, raw_output = _ask_extraction_plan(
                    llm, route, user_query, tables, system_prompt
                )
                payload = _execute_plan(plan, tables, user_query, route)
                planner_outputs.append(raw_output)
                observation = json.dumps(payload, ensure_ascii=False, sort_keys=True)
                status = "PLANNED"
            except (KeyError, TypeError, ValueError) as exc:
                planner_outputs.append(getattr(exc, "planner_output", raw_output))
                errors.append(str(exc))
                payload_tables = []
                for table in tables:
                    frame, columns = table["frame"], list(table["frame"].columns)
                    payload_tables.append({
                        "table_id": f"file_{table['file_index']}_view_0",
                        "file_index": table["file_index"], "file_name": table["path"].name,
                        "role": f"file_{table['file_index']}", "columns": columns,
                        "original_rows": len(frame), "returned_rows": len(frame),
                        "filters": {"logic": "and", "conditions": []},
                        "records": [{"source_row": int(index), "values": {
                            column: str(row[column]) for column in columns
                        }} for index, row in frame.iterrows()],
                    })
                fallback_reason = errors[-1] if errors else "planner did not produce a valid plan"
                observation = json.dumps({
                    "route": route, "query": str(user_query or ""), "tables": payload_tables,
                    "relationships": [], "ignored_file_indices": [],
                    "validation": {
                        "status": "FALLBACK_FULL_DATA", "planner_errors": errors,
                        "fallback_reason": fallback_reason,
                    },
                }, ensure_ascii=False, sort_keys=True)
                status = "FALLBACK_FULL_DATA"
        else:
            observation = rag_answer(tool_query)
        trace = {
            "version": "GPT4.1-LO", "mode": "planned" if planned else "legacy",
            "status": status, "plan": plan, "planner_attempt_count": attempt_count,
            "planner_outputs": planner_outputs, "planner_errors": errors,
            "validation_errors": errors, "fallback_reason": fallback_reason,
            "payload_hash": hashlib.sha256(observation.encode("utf-8")).hexdigest(),
        }
        state.clear()
        state.update({"observation": observation, "trace": trace})
        return observation

    return llm, Tool(name="CSVQA", func=qa_wrapper, description=tool_description), state


def invoke_react_with_required_csvqa(agent, query, route):
    """Require CSVQA while preserving the existing ReAct agent boundary."""
    def call_count(result):
        return sum(
            isinstance(step, tuple)
            and len(step) >= 2
            and getattr(step[0], "tool", None) == "CSVQA"
            for step in result.get("intermediate_steps", [])
        )

    result = agent.invoke(query)
    if call_count(result) == 0:
        result = agent.invoke(
            "Protocol correction: call CSVQA before the Final Answer. Begin with Action: CSVQA.\n\n"
            f"User Description: {query}"
        )
    calls = call_count(result)
    planned = CSVQA_MODE_BY_ROUTE[normalize_route(route)] == "planned"
    if (planned or normalize_route(route) == "RA") and calls != 1:
        raise RuntimeError(f"{route} ReAct must call CSVQA exactly once; observed {calls}")
    if not planned and calls < 1:
        raise RuntimeError(f"{route} legacy ReAct must call CSVQA at least once; observed {calls}")
    return result


def formulate_with_csvqa(query, dataset_address, route, system_prompt, tool_description, prefix, suffix):
    """Run the shared CSVQA agent with the route's prompt and fixed data mode."""
    llm, qa_tool, csvqa_result = build_csvqa_components(
        dataset_address, system_prompt, tool_description, route=route, user_query=query,
    )
    agent = initialize_agent(
        tools=[qa_tool], llm=llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
        agent_kwargs={"prefix": prefix, "suffix": suffix}, verbose=True,
        handle_parsing_errors=True, return_intermediate_steps=True,
    )
    result = invoke_react_with_required_csvqa(agent, query, route)
    return {"formulation": result["output"], "tool_calls": sum(step[0].tool == "CSVQA" for step in result["intermediate_steps"]), **csvqa_result}


def normalize_data_address(value):
    if value is None or (not isinstance(value, str) and pd.isna(value)):
        return ""
    return "\n".join(line.strip() for line in str(value).splitlines() if line.strip())

def csv_schema_preview(dataset_address, query=""):
    """Preview plus full-file evidence; matching never chooses or filters the model data."""
    normalize = lambda value: re.sub(r"\s+", " ", str(value)).strip().casefold()
    normalized_query = normalize(query)
    quoted = re.findall(r'"([^"\n]+)"|“([^”\n]+)”|‘([^’\n]+)’|(?<!\w)\x27([^\x27\n]+)\x27(?!\w)', query)
    names = {normalize(next(part for part in group if part)) for group in quoted}
    blocks = []
    for raw in normalize_data_address(dataset_address).splitlines():
        path = Path(raw).expanduser()
        path = path if path.is_absolute() else PROJECT_ROOT / path
        typed = read_csv_compat(path)
        frame = read_csv_compat(path, dtype=str, keep_default_na=False)
        stats, columns, terms = {}, {}, set(names)
        for column in frame:
            values = frame[column]
            normalized = values.map(normalize)
            columns[column] = normalized
            present = values[normalized.ne("")]
            numbers = pd.to_numeric(present, errors="coerce")
            stats[column] = {"missing": int(normalized.eq("").sum()), "unique_nonempty": int(present.nunique())}
            if len(present) and numbers.notna().all():
                stats[column]["numeric_range"] = [float(numbers.min()), float(numbers.max())]
            # Unquoted evidence must literally name a real column or complete nonnumeric value.
            candidates = [column] + present.unique().tolist()
            for value in candidates:
                term = normalize(value)
                if term and not re.fullmatch(r"[-+\d.,%]+", term) and re.search(r"(?<!\w)" + re.escape(term) + r"(?!\w)", normalized_query):
                    terms.add(term)
        evidence = []
        for term in sorted(terms):
            matches = []
            for column, values in columns.items():
                exact, prefix, contains = values.eq(term), values.str.startswith(term), values.str.contains(term, regex=False)
                if contains.any():
                    matches.append({"column": column, "exact": int(exact.sum()), "prefix": int(prefix.sum()),
                                    "contains": int(contains.sum()), "examples": frame.loc[contains, column].drop_duplicates().head(3).tolist()})
            evidence.append({"term": term, "matching_columns": matches,
                             "exact_matching_columns": sum(item["exact"] > 0 for item in matches)})
        blocks.append(
            f"File: {path.resolve()}\nCSV delimiter: comma; first line consumed as header.\nTotal rows: {len(frame)}\nColumns: {list(frame.columns)}\n"
            f"Parsed column types: {typed.dtypes.astype(str).to_dict()}\n"
            f"Preview only (first 10 rows):\n{frame.head(10).to_string(index=False)}\n"
            f"Full-file column statistics: {json.dumps(stats, ensure_ascii=False)}\n"
            "Matching uses casefold and collapsed/trimmed whitespace; original IDs and leading zeros are preserved.\n"
            "Counts are per column and per individual condition, not intersections. Empty matching_columns means zero matches in every column.\n"
            f"Query-name evidence: {json.dumps(evidence, ensure_ascii=False)}"
        )
    if not blocks:
        raise ValueError("No external CSV paths were supplied")
    return "\n\n---\n\n".join(blocks)


In [ ]:
METHOD='Abl-RAGOnly'

MODEL_RULES = (
    'Use the current question and the retrieved formulation to produce the final answer. '
    'For NRM, use the attached CSVQA structured data as the authoritative source for '
    'current-case values. Do not add information from other cases.\n\n' + OUTPUT_RULES
)

ROUTE_PREFIX = {'NRM': 'You are an assistant that formulates mathematical optimization models.\n'
        '\n'
        '    \n'
        '\n'
        '    \n'
        '\n'
        '    Call CSVQA exactly once and return an ABSTRACT model. Define\n'
        'index sets, parameters, variables, objective, and constraints symbolically. Include a '
        'short Data\n'
        'Mapping that identifies every source by exact table_id and exact column name. Do not copy '
        'record\n'
        'values or literal record counts.\n'
        '    Use the user query as the authority for objective sense, variable domains, boundary '
        'semantics,\n'
        '    and additional constraints. Do not invent missing data or silently omit returned '
        'data.\n'
        '    ',
 'RA': 'You are an assistant that formulates mathematical optimization models.\n'
       '\n'
       '    \n'
       '\n'
       '    \n'
       '\n'
       '    You MUST call CSVQA exactly once before the Final Answer and use all returned rows.\n'
       'Keep the supplied file order and original row order. Never sort or reset an index.\n'
       'Use an explicit business ID column when present; never synthesize business IDs from row '
       'numbers.\n'
       "Do not infer a file's semantic role only from its filename or position.\n"
       'Align profits, resource consumption, and capacities through explicit column/ID keys.\n'
       'Every resource-capacity dimension must receive a capacity constraint, and each\n'
       'activity must appear in the objective and all applicable resource constraints.\n'
       'Infer integer/binary/continuous domains from the user description and data.\n'
       'An item count or number of units to order, produce, allocate, or develop is a nonnegative '
       'integer\n'
       'unless the original query explicitly permits fractional quantities or describes divisible '
       'material.\n'
       'The words scale, quantity or allocation alone do not establish continuous variables.\n'
       'Distinguish a global activity decision from separate resource-by-activity allocations. '
       'When independent\n'
       'warehouses, machines or other resources can each allocate to activities, use decisions '
       'indexed by\n'
       'both resource and activity; do not constrain one identical global expression by every '
       "resource's limit.\n"
       'Each resource-indexed capacity constraint must depend on that resource through its '
       'decisions or\n'
       'consumption coefficients. For different resource dimensions consumed by one global '
       'activity, retain\n'
       'global activity variables and the dimension-specific consumption coefficients instead.\n'
       'Never invent a total capacity, default RHS, or omitted consumption coefficient.\n'
       'Return a complete numerical formulation containing the retrieved identifiers and '
       'coefficients.\n'
       '\n'
       '    Use the user query as the authority for objective sense, variable domains, boundary '
       'semantics,\n'
       '    and additional constraints. Do not invent missing data or silently omit returned '
       'data.\n'
       '    ',
 'TP': 'You are an assistant that formulates mathematical optimization models.\n'
       '\n'
       '    \n'
       '\n'
       '    \n'
       '\n'
       '    Call CSVQA at least once and return a complete numerical formulation. Retrieved\n'
       'Information must contain every identifier and coefficient required by downstream code. '
       'Preserve\n'
       'source order and do not invent, aggregate, or silently omit data.\n'
       '    Use the user query as the authority for objective sense, variable domains, boundary '
       'semantics,\n'
       '    and additional constraints. Do not invent missing data or silently omit returned '
       'data.\n'
       '    ',
 'AP': "You are an assistant that generates a mathematical model based on the user's description "
       'and provided CSV data.\n'
       '\n'
       '            \n'
       '\n'
       '            \n'
       '\n'
       '            Note: Please retrieve all neccessary information from the CSV file to generate '
       'the answer. When you generate the answer, please output required parameters in a whole '
       'text, including all vectors and matrices.\n'
       '\n'
       '            When you need to retrieve information from the CSV file, use the provided '
       'tool.\n'
       '\n'
       '            ',
 'FLP': "You are an assistant that generates a mathematical model based on the user's description "
        'and provided CSV data.\n'
        '\n'
        '            \n'
        '\n'
        '            \n'
        '\n'
        '            Note: Please retrieve all neccessary information from the CSV file to '
        'generate the answer. When you generate the answer, please output required parameters in a '
        'whole text, including all vectors and matrices.\n'
        '\n'
        '            When you need to retrieve information from the CSV file, use the provided '
        'tool.\n'
        '\n'
        '            '}

ROUTE_SUFFIX = {'NRM': '\n\n    Begin!\n\n    User Description: {input}\n    {agent_scratchpad}',
 'RA': '\n\n    Begin!\n\n    User Description: {input}\n    {agent_scratchpad}',
 'TP': '\n\n    Begin!\n\n    User Description: {input}\n    {agent_scratchpad}',
 'AP': '\n'
       '\n'
       '            Begin!\n'
       '\n'
       '            User Description: {input}\n'
       '            {agent_scratchpad}',
 'FLP': '\n'
        '            Begin!\n'
        '\n'
        '            User Description: {input}\n'
        '            {agent_scratchpad}'}

ROUTE_CSVQA_SYSTEM = {'AP': 'Retrieve the documents in order from top to bottom. Use the retrieved context to answer the question. If mention a certain kind of product, retrieve all the relavant product information detail judging by its product name. If not mention a certain kind of product, retrieve all the data instead.Context: {context}', 'FLP': 'Retrieve the documents in order. Use the given context to answer the question. If mention a certain kind of product, retrieve all the relavant product information detail judging by its product name. If not mention a certain kind of product, make sure that all the data is retrieved.Context: {context} Preserve every facility ID, customer ID, FixedCost, Capacity, Demand, and cost-matrix axis together with its source-row position. FixedCost and Capacity must use the same facility ID; the matrix row and column IDs must explicitly match facilities and customers, with its source orientation and shape retained. Model a two-dimensional shipment decision only. Use a stated capacity when present; an absent capacity is unresolved evidence, not zero. Do not transpose, truncate, pad, zero-fill, or infer an extra commodity/product axis.'}

ROUTE_CSVQA_DESCRIPTION = {'AP': 'Use this tool to answer Querys based on the provided CSV data and retrieve product data similar to the input query.', 'FLP': 'Use this tool to answer Querys based on the provided CSV data and retrieve product data similar to the input query.'}

OTHERS_ABSTRACT_PROMPT = '\nYou are an expert optimization modeler.\nYour task is to create an "Abstract Model Plan" based on the user\'s query and the CSV Schema (data structure).\nThis plan is *not* Gurobi code or mathematical formulas, but a clear, step-by-step reasoning process in English.\n\n[Current Task]\nUser Query: {query}\n\nCSV Schema:\n{schema}\n\nAn explicitly enumerated complete entity set in the query is authoritative: extra CSV entities\nmust not enlarge that set unless the query explicitly delegates its definition to the data.\nFor a one-off finite-horizon schedule, change/ramp constraints compare consecutive modeled periods. An initial\nbinary on/off state is not a pre-horizon continuous quantity: do not invent an initial continuous\nlevel or a first-period ramp limit unless the query explicitly supplies that continuous boundary.\nFor recurring daily/cyclic operation, preserve wrap-around coverage across the last/first period\nwhen a working shift spans the boundary; do not impose a fictitious empty start to a recurring day.\nState and implement the original objective sense and complete value expression, including constants.\nDo not offer sign-reversed or constant-shifted optimizer-equivalent objectives as alternatives.\n\n\nData selection: state for each entity table whether all rows or a subset is required; name the\nfilter column, exact condition, entity key, and joins. Preserve shared resource/parameter tables.\nWithout a query-requested filter, use the complete entity set. A preview is never the selected set.\nMatching statistics are evidence, not instructions: do not broaden a zero-match condition to\nsubstring matching or all rows. Check selected keys and applicable counts at runtime; recompute\ncombined conditions rather than treating individual counts as intersection counts.\n\n[Your Output]\nYou must strictly follow this format for your "Abstract Model Plan" output:\n\n[Abstract Model Plan START]\n1.  **Analyze Query:** The user wants to {{{{Your analysis}}}}.\n2.  **Identify Model Type:** Based on the query, this is a {{{{e.g., LP, MIP, Fixed-Charge, Blending}}}} problem.\n3.  **Define Index Sets:** The primary indices are {{{{e.g., Products, Workers, Sources, Destinations}}}}.\n4.  **Define Decision Variables:**\n    -   `x[i]` = {{{{Describe first variable, e.g., \'quantity of product i\'}}}}. Type: {{{{GRB.CONTINUOUS / GRB.INTEGER}}}}.\n    -   `y[i]` = {{{{Describe second variable, e.g., \'if product i is produced\'}}}}. Type: {{{{GRB.BINARY}}}}.\n5.  **Identify Parameters (from Schema):**\n    -   Objective coefficients (e.g., profit) will come from column(s): {{{{e.g., \'Price\', \'Production Cost\'}}}}.\n    -   Constraint coefficients (e.g., resource use) will come from: {{{{e.g., \'Resource 1\', \'Available Time\'}}}}.\n    -   Constraint RHS (limits) will come from: {{{{e.g., \'Max Demand\'}}}}.\n6.  **Formulate Objective:** {{{{Describe objective, e.g., \'Maximize sum((schema[\'Price\'] - schema[\'Cost\']) * x[i] - schema[\'FixedCost\'] * y[i])\'}}}}.\n7.  **Formulate Constraints:**\n    -   Constraint 1 (e.g., Resource Limit): {{{{Describe constraint 1, e.g., \'sum(schema[\'Resource 1\'][i] * x[i]) <= schema[\'Available\']\'}}}}.\n    -   Constraint 2 (e.g., Linking): {{{{Describe constraint 2, e.g., \'x[i] <= M * y[i])\'}}}}.\n    -   ... (Other constraints) ...\n[Abstract Model Plan END]\n'

OTHERS_CODE_PROMPT = '\nYou are an expert Gurobi programmer.\nYour task is to strictly follow the User Query, CSV Schema, and "Abstract Model Plan" to translate them into a single, complete, executable Gurobi Python code block.\nProvide one fenced Python code block.\nThe code must include the necessary imports from gurobipy, pandas, numpy, re, math or json. Do not import sys or os.\nCreate exactly one Gurobi model in m or model and call its optimize() exactly once, both at top level. Do not wrap model creation or solving in functions or try/except; let errors propagate to the execution harness. Do not write files or call computeIIS().\nThe code must read all supplied CSV files needed by the model, at these exact newline-separated paths:\n{dataset_address}\nRead their complete contents at runtime. The schema preview is not the full dataset.\nPreserve explicit business identifiers and matrix axes; do not invent filenames or columns.\nData-reading contract: these files were parsed with pandas.read_csv(path, sep=\',\').\nThe displayed preview is space-aligned for readability; it does not imply a whitespace delimiter.\npandas already consumes the CSV header. Select entity and parameter rows through identifier values\nor validated predicates, never guessed iloc offsets or a guessed number of header rows.\nNormalize identifier types explicitly on both sides of every lookup/join, including matrix row\nindices versus column labels. Preserve the actual identifiers and derive dimensions from all rows.\nSeries.apply receives one cell value, not a row; use vectorized column predicates or DataFrame.apply\nwith axis=1 when a predicate needs a row. Every comprehension must use the indices it binds, not\nstale variables from a preceding loop. Do not silently skip failed conversions or missing coefficients.\nUse setObjective for the exact metric requested by the query, including all additive constants,\nsigns and units. An optimizer-equivalent objective is not an equivalent reported objective value;\ndo not drop constant terms and then merely restore them in printed output.\nThe code must *fully* implement all variables, objectives, and constraints from the "Abstract Model Plan".\n\n[User Query]\n{query}\n\n[CSV Schema]\n{schema}\n\n[Abstract Model Plan]\n{abstract_plan}\n\nAn explicitly enumerated complete entity set in the query is authoritative: extra CSV entities\nmust not enlarge that set unless the query explicitly delegates its definition to the data.\nFor a one-off finite-horizon schedule, change/ramp constraints compare consecutive modeled periods. An initial\nbinary on/off state is not a pre-horizon continuous quantity: do not invent an initial continuous\nlevel or a first-period ramp limit unless the query explicitly supplies that continuous boundary.\nFor recurring daily/cyclic operation, preserve wrap-around coverage across the last/first period\nwhen a working shift spans the boundary; do not impose a fictitious empty start to a recurring day.\nState and implement the original objective sense and complete value expression, including constants.\nDo not offer sign-reversed or constant-shifted optimizer-equivalent objectives as alternatives.\n\n\nData selection: state for each entity table whether all rows or a subset is required; name the\nfilter column, exact condition, entity key, and joins. Preserve shared resource/parameter tables.\nWithout a query-requested filter, use the complete entity set. A preview is never the selected set.\nMatching statistics are evidence, not instructions: do not broaden a zero-match condition to\nsubstring matching or all rows. Check selected keys and applicable counts at runtime; recompute\ncombined conditions rather than treating individual counts as intersection counts.\n\nReserve m and model for the Gurobi model; never reuse them as loop variables or other values.\nKeep solver display names short and separate from business keys. With long keys, create variables\nindividually under short display names and store them in a dictionary keyed by the original IDs;\ndo not let addVars expand long business keys into solver names. Apply the same rule to constraint names.\n\n[Your Answer]\nGive an LP or MILP mathematical model and complete Gurobi code in the required two-section format.\n'

def classify(client,case):
    return plain_call(client,CLASS_RULES,case['query'])

def answer_others(client,case):
    # The full workflow uses a schema preview and two model calls for this route.
    addresses='\n'.join(rel(path) for path in case['files'])
    schema=csv_schema_preview(addresses,query=case['query'])
    plan_prompt=OTHERS_ABSTRACT_PROMPT.format(
        query=case['query'],schema=schema)
    plan,plan_usage=plain_call(
        client,'Produce an abstract optimization model plan from the current question and CSV schema.',
        plan_prompt)
    code_prompt=OTHERS_CODE_PROMPT.format(
        query=case['query'],schema=schema,abstract_plan=plan,
        dataset_address=addresses)
    answer,code_usage=plain_call(client,MODEL_RULES,code_prompt)
    trace={'status':'LEGACY_SCHEMA','planner_attempt_count':0,
           'payload_hash':hashlib.sha256(schema.encode()).hexdigest()}
    return answer,plan_usage+code_usage,0,[trace]

def answer_case(client,case,predicted):
    category=route(predicted)
    if category=='Others':
        return answer_others(client,case)
    planned=CSVQA_MODE_BY_ROUTE[category]=='planned'
    system_prompt=(CSVQA_PLANNED_PROMPTS[category] if planned
                   else ROUTE_CSVQA_SYSTEM.get(category,CSVQA_LEGACY_SYSTEM_PROMPT))
    description=(CSVQA_TOOL_DESCRIPTIONS[category] if planned
                 else ROUTE_CSVQA_DESCRIPTION.get(category,CSVQA_LEGACY_TOOL_DESCRIPTION))
    prefix=ROUTE_PREFIX[category]
    suffix=ROUTE_SUFFIX[category]
    addresses='\n'.join(rel(path) for path in case['files'])
    result=formulate_with_csvqa(
        case['query'],addresses,category,system_prompt,description,prefix,suffix)
    user=(
        'Current question:\n'+case['query']+
        '\n\nCurrent case CSV paths:\n'+addresses+
        '\n\nCSVQA-based formulation:\n'+result['formulation']
    )
    if planned:
        user+='\n\nCSVQA structured data for this NRM case:\n'+result['observation']
    answer,api_usage=plain_call(client,MODEL_RULES,user)
    return answer,api_usage,result['tool_calls'],[result['trace']]

def run(rows=None,output_dir=None):
    from dotenv import load_dotenv
    from openai import OpenAI
    load_dotenv(ROOT/'.env.local',override=False)
    load_dotenv(ROOT/'.env',override=False)
    if not os.environ.get('OPENAI_API_KEY'):raise RuntimeError('OPENAI_API_KEY is missing')
    require_rag_dependencies()
    selected=set(range(101) if rows is None else rows)
    if not selected.issubset(set(range(101))):raise ValueError('Rows must be 0..100')
    all_cases=cases()
    stamp=datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
    folder=Path(output_dir or ROOT/'outputs'/'large_scale_or_ablations'/f'ragonly__{stamp}').resolve()
    folder.mkdir(parents=True,exist_ok=True)
    (folder/'answers').mkdir(exist_ok=True)
    log=folder/'records.jsonl'
    done={json.loads(x)['row'] for x in log.read_text().splitlines() if x.strip()} if log.exists() else set()
    client=OpenAI(max_retries=0,timeout=180)
    for case in all_cases:
        i=case['row']
        if i not in selected or i in done:continue
        rec={'row':i,'gold_type':gold_class(case['type']),'gold_objective':case['gold'],
             'files':[rel(p) for p in case['files']],'status':'error','tool_calls':0}
        t=time.perf_counter()
        try:
            label,cls_usage=classify(client,case)
            pred=label.strip().strip('`*_ .,:;\n')
            rec['pred_type']=pred
            rec['classification_correct']=pred==rec['gold_type']
            answer,model_usage,ncalls,traces=answer_case(client,case,pred)
            rec['rag_trace']=traces
            rec['tool_calls']=ncalls
            rec['usage']={k:sum(x[k] for x in cls_usage+model_usage) for k in ('input','output')}
            (folder/'answers'/f'row_{i:03d}.txt').write_text(answer,encoding='utf-8')
            source=extract_code(answer)
            objective,solver=solve_code(source)
            rec.update(status='ok',predicted_objective=objective,
                correct_optimal=math.isclose(objective,case['gold'],rel_tol=1e-4,abs_tol=1e-4),
                solver_tail=solver)
        except Exception as exc:
            rec.update(correct_optimal=False,error_type=type(exc).__name__,error=str(exc)[-3000:])
        rec['wall_s']=round(time.perf_counter()-t,3)
        with log.open('a',encoding='utf-8') as f:f.write(json.dumps(rec,ensure_ascii=False)+'\n')
        print('row',i,rec['status'],'correct',rec['correct_optimal'])
    save_scores(folder,selected)
    return folder


## Preflight

This reads the benchmark and checks every case file without calling an API.

In [ ]:
DATA = check_files()
print("Model:", MODEL, "Method:", METHOD)


## Run

Set `RUN_API` to `True` after choosing rows. Use `ROWS = None` for all 101 cases. Reuse `OUTPUT_DIR` to resume.

In [ ]:
RUN_API = False
ROWS = None
OUTPUT_DIR = None
if RUN_API:
    RUN_DIR = run(rows=ROWS, output_dir=OUTPUT_DIR)
    print(RUN_DIR)
